In [633]:
import pandas as pd
import numpy as np
from pathlib import Path


import altair as alt

In [634]:
root = "/Volumes/ArxivE/_Tobias/LSM980/Mitotic_Stopwatch/DataFrames/"

df = pd.read_csv(root + "MainDataFrame_MitoticStopwatch_Monastrol.csv")

In [635]:
#df = df[df["Dataset"] == 20250627]

In [636]:
colourscheme = "viridis"

In [637]:
def Binned_Circle2(
    data, x, y, x_title, y_title, color, column, 
    Circlesize = 125,  
    Bin_Boxplot_width = 100, 
    Bin_Boxplot_height = 150,
    RawPointSize = 18  # Size for individual points
):

    # Shared encodings
    base_x = alt.X(x, title = None, axis = alt.Axis(grid = False))
    base_color = alt.Color(color, scale = alt.Scale(scheme = colourscheme))

    # Chart 1: Mean Circle
    mean = alt.Chart(data, width = Bin_Boxplot_width, height = Bin_Boxplot_height).mark_circle(size = Circlesize).encode(
        x = base_x,
        y = alt.Y(f"mean({y})", title = y_title, axis = alt.Axis(grid = False)),
        color = base_color
    )

    # Chart 2: Mean Error Bar
    mean_sem = alt.Chart(data, width = Bin_Boxplot_width, height = Bin_Boxplot_height).mark_errorbar().encode(
        x = base_x,
        y = alt.Y(f"mean({y})", title = y_title, axis = alt.Axis(grid = False)),
        color = base_color
    )

    # Chart 3: Raw Data Points
    raw_points = alt.Chart(data, width = Bin_Boxplot_width, height = Bin_Boxplot_height).mark_circle(size = RawPointSize, opacity = 0.1).encode(
        x = base_x,
        y = alt.Y(y, title = y_title, axis = alt.Axis(grid = False)),
        color = base_color
    )

    # Layer all three
    layered = raw_points + mean_sem + mean

    # Apply facet to combined chart
    faceted = layered.facet(
        column = column,
        spacing = 15
    )

    # Configure chart
    return faceted.configure_axis(
                grid = True, ticks = True, labelPadding = 5, offset = 12
           ).configure_header(
                labelOrient = 'bottom', title = None
           ).configure_view(
                stroke = 'transparent', 
                strokeWidth = 0.5
           )

In [638]:
def Scatter_bin(dataframe, x, y, color, x_title, y_title, binextent, binstep,
            Circlesize = 15, 
            CircleOpacity = 0.2,  
            Scatter_width = 250, 
            Scatter_height = 200
               ):
    # Standard scatter plot 
    Scatter = alt.Chart(
        data = dataframe, 
        width = Scatter_width, 
        height = Scatter_height
    ).mark_circle(
        opacity = CircleOpacity,
        size = Circlesize
    ).encode(
        alt.X(x, title = x_title),
        alt.Y(y, title = y_title),
        color = alt.Color(
            color, legend = None, scale = alt.Scale(scheme = colourscheme)
        ) 
    )
    
    Scatter_binned = alt.Chart(
        data = dataframe, 
        width = Scatter_width, 
        height = Scatter_height
    ).mark_circle(
        opacity = 1,
        size = 100
    ).encode(
        alt.X(x, title = x_title, bin = alt.Bin(extent = binextent, step = binstep)),
        alt.Y(f'mean({y})', title = y_title, bin = False),
        color = alt.Color(
            color, scale = alt.Scale(scheme = colourscheme)
        ) 
    )
    
    Error_Scatterbinned = alt.Chart(
            data = dataframe
    ).mark_errorbar(extent = "ci").encode(
        alt.X(x, title = x_title, bin = alt.Bin(extent = binextent, step = binstep)),
        alt.Y(f'mean({y})', title = y_title, bin = False),
        color = alt.Color(
            color, scale = alt.Scale(scheme = colourscheme)
        ) 
    ) 
    SCATTERBIN = Error_Scatterbinned + Scatter_binned
    return SCATTERBIN

In [639]:
Scatter_MitoticDuration = Scatter_bin(
    dataframe = df,
    x = "Frame_onset_hrs",
    y = "Mitotic_duration_mins",
    color = "Splitting_event",
    x_title = "Time (h)",
    y_title = "Mitotic duration (min)",
    binextent = [0, 72],
    binstep = 6
)
Scatter_MitoticDuration

alt.LayerChart(...)

In [640]:
df.shape

(492, 49)

In [641]:
Bincircle_arrest = Binned_Circle2(
    data = df[df["Cell_Cycle_hours"] > 1],
    x = 'Mother_arrested',
    y = 'Cell_Cycle_hours',
    x_title = '',
    y_title = 'Cell_Cycle_hours',
    color = 'Mother_arrested:O',
    column = 'Splitting_event'
)
Bincircle_arrest

alt.FacetChart(...)

In [642]:
df.groupby("Mother_arrested").Cell_Cycle_hours.mean()

Mother_arrested
000-030 min    32.200000
030-060        27.676415
060-090        29.346970
090-120        39.520833
> 120 min      48.412179
Name: Cell_Cycle_hours, dtype: float64

In [643]:
df.Mother_arrested.value_counts()

Mother_arrested
030-060        159
060-090         77
> 120 min       52
090-120         12
000-030 min      5
Name: count, dtype: int64

In [681]:
def plot_Individual_Daughters(df, barwidth = 2, plotwidth = 700, plotheight = 200):
    bar = alt.Chart(df).mark_bar(width = barwidth).encode(
        x = alt.X(
            "Unique_Source_ID:N",
            title = "Individual daughter cells",
            sort = alt.EncodingSortField(
                field = "Mother_Mitotic_duration_mins",
                order = "ascending"
            ),
            axis = alt.Axis(
                labels = False, 
                ticks = False
            )
        ),
        #y = alt.Y("Mother_Mitotic_duration_mins:Q", title = "Mother mitotic duration (min)", bin = alt.Bin(maxbins = 30)),
        y = alt.Y("Mother_Mitotic_duration_mins:Q", title = "Mother mitotic duration (min)", scale = alt.Scale(domain = [0, 700])),
        color = alt.Color("Cell_Cycle_hours:Q", title = "Daughter cell cycle duration (hours)", scale = alt.Scale(scheme = "brownbluegreen")),
    )

    chart = bar.properties(
        width = plotwidth,
        height = plotheight,
    )

    return chart

In [683]:
# All data without selection!

daughters = plot_Individual_Daughters(df[df["Mother_Mitotic_duration_mins"] > 0], barwidth = 2, plotwidth = 600, plotheight = 250)
daughters

alt.Chart(...)

In [646]:
df.Unique_Track_ID.nunique()

183

In [647]:
# TO DO filter out tracks or branches that wander out of FOV

gen1_high_mitotic = df[(df["Generation"] == 1) & (df["Mitotic_duration_mins"] > 100)]

track_ids_of_interest = gen1_high_mitotic["Unique_Track_ID"].unique()

result_df = df[df["Unique_Track_ID"].isin(track_ids_of_interest)]

print(f"Number of Track_IDs (unique) with Generation 1 Mitotic_duration_mins > 100: {len(track_ids_of_interest)}")

Number of Track_IDs (unique) with Generation 1 Mitotic_duration_mins > 100: 81


In [648]:
# Get the unique Track_IDs in result_df
all_track_ids = result_df["Unique_Track_ID"].unique()
total_track_ids = len(all_track_ids)

# Group by Track_ID and get the max Generation per track
max_gen_by_track = result_df.groupby("Unique_Track_ID")["Generation"].max()

max_gen_by_track.head()

Unique_Track_ID
2025062730    2
2025062731    1
2025062732    2
2025062737    2
2025062738    1
Name: Generation, dtype: int64

In [649]:
# a) Only Generation = 1
only_gen1 = (max_gen_by_track == 1).sum()
print(f"Track_IDs with max generation 1: {only_gen1}")

# b) Only Generations ≤ 2
only_gen2 = (max_gen_by_track == 2).sum()
print(f"Track_IDs with max generation 2: {only_gen2}")

# c) Only Generations ≤ 3
only_gen3 = (max_gen_by_track == 3).sum()
print(f"Track_IDs with max generation 3: {only_gen3}")

# Compute percentages
percent_only_gen1 = (only_gen1 / total_track_ids) * 100
percent_only_gen2 = (only_gen2 / total_track_ids) * 100
percent_only_gen3 = (only_gen3 / total_track_ids) * 100

# Print results
print(f"Total Track_IDs: {total_track_ids}")
print(f"Percentage with ONLY Generation = 1: {percent_only_gen1:.2f}%")
print(f"Percentage with Generation ≤ 2: {percent_only_gen2:.2f}%")
print(f"Percentage with Generation ≤ 3: {percent_only_gen3:.2f}%")

Track_IDs with max generation 1: 39
Track_IDs with max generation 2: 37
Track_IDs with max generation 3: 4
Total Track_IDs: 81
Percentage with ONLY Generation = 1: 48.15%
Percentage with Generation ≤ 2: 45.68%
Percentage with Generation ≤ 3: 4.94%


In [650]:
def compute_generation_percentages(df):
    # Helper to compute percentages by dataset
    def calculate_percentages(sub_df, population_label):
        # Get max generation per Track_ID
        grouped = sub_df.groupby(["Unique_Track_ID", "Dataset"])["Generation"].max().reset_index()

        # Compute category flags
        grouped["Max_Gen1"] = grouped["Generation"] == 1
        grouped["Max_Gen2"] = grouped["Generation"] == 2
        grouped["Max_Gen3"] = grouped["Generation"] == 3
        grouped["Population"] = population_label

        # Aggregate counts and percentages by Dataset
        summary = grouped.groupby(["Dataset", "Population"]).agg(
            Total_Tracks = ("Unique_Track_ID", "count"),
            Tracks_Only_Gen1 = ("Max_Gen1", "sum"),
            Tracks_Only_Gen2 = ("Max_Gen2", "sum"),
            Tracks_Only_Gen3 = ("Max_Gen3", "sum")
        ).reset_index()

        # Calculate percentages
        summary["Max_Gen1_percent"] = 100 * summary["Tracks_Only_Gen1"] / summary["Total_Tracks"]
        summary["Max_Gen2_percent"] = 100 * summary["Tracks_Only_Gen2"] / summary["Total_Tracks"]
        summary["Max_Gen3_percent"] = 100 * summary["Tracks_Only_Gen3"] / summary["Total_Tracks"]

        return summary[[
            "Dataset", "Population", "Total_Tracks",
            "Max_Gen1_percent", "Max_Gen2_percent", "Max_Gen3_percent"
        ]]

    # Mother cell prolonged mitotic arrest
    gen1_high = df[(df["Generation"] == 1) & (df["Mitotic_duration_mins"] > 100)]
    high_track_ids = gen1_high["Unique_Track_ID"].unique()
    df_high = df[df["Unique_Track_ID"].isin(high_track_ids)]
    summary_high = calculate_percentages(df_high, "High_Mitotic")

    # Mother cell little or no mitotic arrest
    gen1_low = df[(df["Generation"] == 1) & (df["Mitotic_duration_mins"] < 100)]
    low_track_ids = gen1_low["Unique_Track_ID"].unique()
    df_low = df[df["Unique_Track_ID"].isin(low_track_ids)]
    summary_low = calculate_percentages(df_low, "Low_Mitotic")

    # Combine results
    final_summary = pd.concat([summary_high, summary_low], ignore_index = True)

    # Reshape to long format
    final_long = final_summary.melt(
        id_vars = ["Dataset", "Population", "Total_Tracks"],
        value_vars = ["Max_Gen1_percent", "Max_Gen2_percent", "Max_Gen3_percent"],
        var_name = "Metric",
        value_name = "Percentage"
    )

    return final_long

percentages_df_all = compute_generation_percentages(df)

percentages_df_all.to_csv(root + "PercentagesDataFrame_MitoticStopwatch_Monastrol_all.csv")

percentages_df_all

,Dataset,Population,Total_Tracks,Metric,Percentage
0,20250627,High_Mitotic,43,Max_Gen1_percent,51.162791
1,20250704,High_Mitotic,38,Max_Gen1_percent,44.736842
2,20250627,Low_Mitotic,32,Max_Gen1_percent,18.750000
3,20250704,Low_Mitotic,70,Max_Gen1_percent,25.714286
4,20250627,High_Mitotic,43,Max_Gen2_percent,41.860465
5,20250704,High_Mitotic,38,Max_Gen2_percent,50.000000
6,20250627,Low_Mitotic,32,Max_Gen2_percent,37.500000
7,20250704,Low_Mitotic,70,Max_Gen2_percent,37.142857
8,20250627,High_Mitotic,43,Max_Gen3_percent,6.976744
9,20250704,High_Mitotic,38,Max_Gen3_percent,2.631579


In [651]:
def plot_percentage_comparison(df_long):

    bar = alt.Chart(df_long).mark_bar(width = 15).encode(
        x = alt.X("Population:N", title = "", axis = alt.Axis(labelAngle = 0)),
        y = alt.Y("mean(Percentage):Q", title = "% of cell families"),
        color = alt.Color("Metric:N", title = "Metric", scale = alt.Scale(scheme = "viridis")),
        xOffset = "Metric:N"
    )

    error = alt.Chart(df_long).mark_errorbar().encode(
        x = alt.X("Population:N"),
        y = alt.Y("mean(Percentage):Q", title = "% of cell families"),
        color = alt.Color("Metric:N"),
        xOffset = "Metric:N"
    )

    datasets = alt.Chart(df_long).mark_circle(width = 15).encode(
        x = alt.X("Population:N", title = "", axis = alt.Axis(labelAngle = 0)),
        y = alt.Y("Percentage:Q", title = "% of cell families"),
        color = alt.Color("Dataset:N", title = "Metric", scale = alt.Scale(scheme = "viridis")),
        xOffset = "Metric:N"
    )

    chart = (bar + datasets + error).properties(
        width = 200,
        height = 200,
    )

    return chart

In [652]:
# All data, without selection

chart_all = plot_percentage_comparison(percentages_df_all)
chart_all

alt.LayerChart(...)

In [653]:
def plot_number_of_tracks(df_long):

    bar = alt.Chart(df_long).mark_bar(width = 15).encode(
        x = alt.X("Population:N", title = "", axis = alt.Axis(labelAngle = -45)),
        y = alt.Y("mean(Total_Tracks):Q", title = "Nr of tracked families"),
       # color = alt.Color("Metric:N", title = "Metric", scale = alt.Scale(scheme = "viridis")),
       # xOffset = "Metric:N"
    )

    error = alt.Chart(df_long).mark_errorbar().encode(
        x = alt.X("Population:N"),
        y = alt.Y("mean(Total_Tracks):Q", title = "Nr of tracked families"),
       # color = alt.Color("Metric:N"),
       # xOffset = "Metric:N"
    )

    datasets = alt.Chart(df_long).mark_circle().encode(
        x = alt.X("Population:N"),
        y = alt.Y("Total_Tracks:Q", title = "Nr of tracked families"),
        color = alt.Color("Dataset:N", scale = alt.Scale(scheme = "viridis")),
       # xOffset = "Metric:N"
    )

    chart = (bar + datasets + error).properties(
        width = 75,
        height = 150,
    )

    return chart

In [654]:
#chart = plot_number_of_tracks(percentages_df)
#chart

In [655]:
# Compute the maximum generation per Unique_Track_ID
max_gen_per_track = df.groupby("Unique_Track_ID")["Generation"].max()

# Create a mapping of Track_ID to Daughter_dividing? status
daughter_dividing_map = (max_gen_per_track > 1)

# Map the status back to the original df
df["Daughter_dividing?"] = df["Unique_Track_ID"].map(daughter_dividing_map)


In [656]:
df["Daughter_dividing?"] = df["Daughter_dividing?"].astype(str)  # ensure it's a string for coloring

def plot_Individual_Daughters2(df, plotwidth = 800, barwidth = 2):
    bar = alt.Chart(df).mark_bar(width = barwidth).encode(
        x = alt.X(
            "Unique_Source_ID:N",
            title = "Individual daughter cells",
            sort = alt.EncodingSortField(
                field = "Mitotic_duration_mins",
                order = "ascending"
            ),
            axis = alt.Axis(
                labels = False, 
                ticks = False
            )
        ),
     #   y = alt.Y("Mitotic_duration_mins:Q", title = "Mother mitotic duration (min)", bin = alt.Bin(maxbins = 30)),
        y = alt.Y("Mitotic_duration_mins:Q", title = "Mother mitotic duration (min)"),
        color = alt.Color("Daughter_dividing?", title = "Daughter dividing?", scale = alt.Scale(scheme = "viridis")),
    )

    chart = bar.properties(
        width = plotwidth,
        height = 200,
    )

    return chart
daughters2 = plot_Individual_Daughters2(df[(df["Mitotic_duration_mins"] > 0) & (df["Mitotic_duration_mins"] < 1000)])
daughters2

alt.Chart(...)

In [657]:
# Select cutoff manually
Threshold_manual = 90

# Select cropping for cohort
cohort_min = 0 # hours
cohort_max = 18.1 # hours


# Function to find out if mother divided within selection time (cohort max)

def mother_divides_in_selection(x, dataframe = df, max_time = cohort_max):
    if x.Generation < 2:
        return False
    else:
        mother_row = dataframe.loc[
            (dataframe.Dataset == x.Dataset) &
            (dataframe.Position == x.Position) &
            (dataframe.Source_ID == x.Mother_ID)
        ]
        mother_time = mother_row["Frame_hrs"]

        if mother_row["Frame_hrs"].shape[0] == 1:   
            if mother_time.item() < max_time:
                return True
            else:
                return False
        else:
            return False

df["Mother_divides_in_time"] = df.apply(mother_divides_in_selection, axis = 1)


print(df["Unique_Track_ID"].nunique())

print(df[(df["Frame_hrs"] < cohort_max) & (df["Frame_onset_hrs"] > cohort_min)].Unique_Track_ID.nunique())

print(df[(df["Frame_hrs"] < cohort_max) & (df["Frame_onset_hrs"] > cohort_min) & (df["Generation"] == 1)].Unique_Track_ID.nunique())

183
142
142


In [658]:
# copied from augmin analysis

def compute_generation_percentages(df, depletion_time_min = cohort_min, depletion_time_max = cohort_max, threshold = Threshold_manual):
    
    # Helper to compute percentages by dataset
    def calculate_percentages(sub_df, population_label):
        
        # Get max generation per Track_ID
        grouped = sub_df.groupby(["Unique_Track_ID", "Dataset"])["Generation"].max().reset_index()

        # Compute category flags
        grouped["Max_Gen1"] = grouped["Generation"] == 1
        grouped["Max_Gen2"] = grouped["Generation"] == 2
        grouped["Max_Gen3"] = grouped["Generation"] == 3
        grouped["Population"] = population_label

        # Aggregate counts and percentages by Dataset
        summary = grouped.groupby(["Dataset", "Population"]).agg(
            Total_Tracks = ("Unique_Track_ID", "count"),
            Tracks_Only_Gen1 = ("Max_Gen1", "sum"),
            Tracks_Only_Gen2 = ("Max_Gen2", "sum"),
            Tracks_Only_Gen3 = ("Max_Gen3", "sum")
        ).reset_index()

        # Calculate percentages
        summary["Max_Gen1_percent"] = 100 * summary["Tracks_Only_Gen1"] / summary["Total_Tracks"]
        summary["Max_Gen2_percent"] = 100 * summary["Tracks_Only_Gen2"] / summary["Total_Tracks"]
        summary["Max_Gen3_percent"] = 100 * summary["Tracks_Only_Gen3"] / summary["Total_Tracks"]

        return summary[[
            "Dataset", "Population", "Total_Tracks",
            "Max_Gen1_percent", "Max_Gen2_percent", "Max_Gen3_percent"
        ]]

    print(f"The number of total tracks before selecting: {df["Unique_Track_ID"].nunique()}")
    
    # Mother cell prolonged mitotic arrest
    gen1_high = df[
        (df["Generation"] == 1) & 
        (df["Mitotic_duration_mins"] > threshold) & 
        (df["Frame_hrs"] < depletion_time_max) &
        (df["Frame_onset_hrs"] > depletion_time_min)
    ]
    high_track_ids = gen1_high["Unique_Track_ID"].unique()
    print(f"The number of selected tracks that have HIGH mitotic times: {len(high_track_ids)}")
    df_high = df[df["Unique_Track_ID"].isin(high_track_ids)]
    summary_high = calculate_percentages(df_high, f">{threshold} min") # calculating percentages using the helper function

    # Mother cell little or no mitotic arrest
    gen1_low = df[
        (df["Generation"] == 1) & 
        (df["Mitotic_duration_mins"] < threshold) & 
        (df["Frame_hrs"] < depletion_time_max) &
        (df["Frame_onset_hrs"] > depletion_time_min)
    ]
    low_track_ids = gen1_low["Unique_Track_ID"].unique()
    print(f"The number of selected tracks that have LOW mitotic times: {len(low_track_ids)}")
    df_low = df[df["Unique_Track_ID"].isin(low_track_ids)]
    summary_low = calculate_percentages(df_low, f"<{threshold} min") # calculating percentages using the helper function

    # Combine results
    final_summary = pd.concat([summary_high, summary_low], ignore_index = True)

    # Reshape to long format for plotting
    final_long = final_summary.melt(
        id_vars = ["Dataset", "Population", "Total_Tracks"],
        value_vars = ["Max_Gen1_percent", "Max_Gen2_percent", "Max_Gen3_percent"],
        var_name = "Metric",
        value_name = "Percentage"
    )

    return df_high, df_low, final_long

results_df_high, results_df_low, percentages_df = compute_generation_percentages(df)
results_df_high_noError, results_df_low_noError, percentages_df_noError = compute_generation_percentages(df[df['any_true'] != True])

percentages_df.to_csv(root + "PercentagesDataFrame_MitoticStopwatch_Monastrol_SelectedCohort.csv")
#percentages_df_noError.to_csv(root + "PercentagesDataFrame_MitoticStopwatch_Monastrol_SelectedCohort_NoErrors.csv")

The number of total tracks before selecting: 183
The number of selected tracks that have HIGH mitotic times: 81
The number of selected tracks that have LOW mitotic times: 61
The number of total tracks before selecting: 171
The number of selected tracks that have HIGH mitotic times: 67
The number of selected tracks that have LOW mitotic times: 56


In [659]:
def plot_percentage_comparison(x = "Population:N", 
                               color = "Dataset:O",
                               color_bar = "Metric:N",
                               offset = "Metric:N", 
                               y = "Percentage", 
                               y_title = "% of cell families", 
                               data = percentages_df
                              ):

    bar = alt.Chart(data).mark_bar(width = 15).encode(
        x = alt.X(x, title = "", axis = alt.Axis(labelAngle = -0)),
        y = alt.Y(f"mean({y}):Q", title = y_title),
        color = alt.Color(color_bar, title = "", scale = alt.Scale(scheme = "viridis")),
        xOffset = offset
    )

    error = alt.Chart(data).mark_errorbar(extent = "stderr").encode(
        x = alt.X(x),
        y = alt.Y(f"mean({y}):Q", title = y_title),
        color = alt.Color(color_bar),
        xOffset = offset
    )

    datasets = alt.Chart(data).mark_circle(width = 15).encode(
        x = alt.X(x, title = "", axis = alt.Axis(labelAngle = 0)),
        y = alt.Y(f"mean({y})", title = y_title),
        color = alt.Color(color, title = "", scale = alt.Scale(scheme = "viridis")),
        xOffset = offset
    )

    chart = (bar + datasets + error).properties(
        width = 200,
        height = 200,
    )

    return chart

In [660]:
# TODO find n of cells in selection

chart_monastrol = plot_percentage_comparison(
    data = percentages_df
)

chart_monastrol

alt.LayerChart(...)

In [661]:
# with errors filtered 

chart_monastrol_noerrors = plot_percentage_comparison(
    data = percentages_df_noError
)

chart_monastrol_noerrors

alt.LayerChart(...)

In [662]:
result_df = pd.concat([results_df_high, results_df_low])

# Compute the maximum generation per Unique_Track_ID
max_gen_per_track = result_df.groupby("Unique_Track_ID")["Generation"].max()

# Create a mapping of Track_ID to Daughter_dividing? status
daughter_dividing_map = (max_gen_per_track > 1)

# Map the status back to the original df
result_df["Daughter_dividing?"] = result_df["Unique_Track_ID"].map(daughter_dividing_map)

result_df["Daughter_dividing?"] = result_df["Daughter_dividing?"].astype(str)  # ensure it's a string for coloring

result_df = result_df[result_df["Frame_hrs"] < cohort_max] 

result_df.to_csv(root + "Filtered_Monastrol.csv")

In [663]:

df.any_true.value_counts()

any_true
False    188
True      22
Name: count, dtype: int64

In [664]:
# Selected cohort
result_df.any_true.value_counts()

any_true
False    116
True      19
Name: count, dtype: int64

In [665]:
# filter out errors

df_filtered = df[df["any_true"] != True]
result_df_filtered = result_df[result_df["any_true"] != True]

result_df_filtered.any_true.value_counts()

any_true
False    116
Name: count, dtype: int64

In [666]:
result_df_filtered.groupby("Mitotic_duration_bin").any_true.value_counts()

Mitotic_duration_bin  any_true
(0, 30]               False        1
(120, 150]            False        8
(150, 180]            False        2
(180, 210]            False        5
(210, 240]            False        5
(240, 270]            False        7
(270, 300]            False        2
(30, 60]              False       24
(300, 330]            False        5
(330, 360]            False        1
(360, 390]            False        1
(390, 420]            False        8
(420, 450]            False        4
(450, 480]            False        9
(60, 90]              False       22
(630, 660]            False        1
(660, 690]            False        1
(90, 120]             False        7
Name: count, dtype: int64

In [667]:
daughters_selected = plot_Individual_Daughters2(result_df, plotwidth = 350, barwidth = 2.5)
daughters_selected

alt.Chart(...)

In [668]:
daughters_selected_no_errors = plot_Individual_Daughters2(result_df_filtered, plotwidth = 350, barwidth = 2.8)
daughters_selected_no_errors

alt.Chart(...)

In [669]:
daughters = plot_Individual_Daughters(
    df[df["Mother_divides_in_time"] == True], 
    barwidth = 2.5, 
    plotwidth = 350
)
daughters

alt.Chart(...)

In [670]:
 df[df["Mother_divides_in_time"] == True].shape

(147, 51)

In [671]:
daughters_noError = plot_Individual_Daughters(
    df_filtered[df_filtered["Mother_divides_in_time"] == True], 
    barwidth = 2.5, 
    plotwidth = 350
)
daughters_noError

alt.Chart(...)

In [672]:
 df_filtered[df_filtered["Mother_divides_in_time"] == True].shape

(144, 51)

In [673]:
def Scatter_bin2(dataframe, x, y, color, x_title, y_title, binextent, binstep,
            Circlesize = 20, 
            CircleOpacity = 0.2,  
            Scatter_width = 150, 
            Scatter_height = 150
               ):
    # Standard scatter plot 
    Scatter = alt.Chart(
        data = dataframe, 
        width = Scatter_width, 
        height = Scatter_height
    ).mark_circle(
        opacity = CircleOpacity,
        size = Circlesize
    ).encode(
        alt.X(x, title = x_title),
        alt.Y(y, title = y_title),
        color = alt.Color(
            color, legend = None, scale = alt.Scale(scheme = colourscheme)
        ) 
    )
    
    Scatter_binned = alt.Chart(
        data = dataframe, 
        width = Scatter_width, 
        height = Scatter_height
    ).mark_circle(
        opacity = 1,
        size = 100
    ).encode(
        alt.X(x, title = x_title, bin = alt.Bin(extent = binextent, step = binstep)),
        alt.Y(f'mean({y})', title = y_title, bin = False),
        color = alt.Color(
            color, scale = alt.Scale(scheme = colourscheme)
        ) 
    )
    
    Error_Scatterbinned = alt.Chart(
            data = dataframe
    ).mark_errorbar(extent = "ci").encode(
        alt.X(x, title = x_title, bin = alt.Bin(extent = binextent, step = binstep)),
        alt.Y(f'mean({y})', title = y_title, bin = False),
        color = alt.Color(
            color, scale = alt.Scale(scheme = colourscheme)
        ) 
    ) 
    SCATTERBIN = Error_Scatterbinned + Scatter_binned
    return SCATTERBIN

In [674]:
Scatter_MitoticDuration_CC = Scatter_bin2(
    dataframe = df_filtered[df_filtered["Mother_divides_in_time"] == True],
    y = "Cell_Cycle_hours",
    x = "Mother_Mitotic_duration_mins",
    color = "Splitting_event",
    y_title = "Daughter Cell cycle duration (h)",
    x_title = "Mother Mitotic duration (min)",
    binextent = [0, 240],
    binstep = 60
)
Scatter_MitoticDuration_CC

alt.LayerChart(...)

In [675]:
df.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'Track_ID', 'Source_ID',
       'Splitting_event', 'Lineage', 'Track_Coordinate_X',
       'Track_Coordinate_Y', 'Frame', 'Position', 'Dataset', 'Generation',
       'Mother_ID', 'Grandmother_ID', 'Sister_ID', 'Cell_Cycle_mins',
       'Cell_Cycle_hours', 'Seen_sister', 'Random_ID', 'Seen_granny', ' ',
       'Label', 'X', 'Y', 'Frame_onset', 'Frames_in_mitosis',
       'Mitotic_duration_mins', 'Interphase_duration_mins',
       'Interphase_duration_hrs', 'Mitotic_duration_hrs',
       'Mother_Mitotic_duration_mins', 'Mother_arrested', 'Metaphase_arrested',
       'Frame_onset_hrs', 'Frame_hrs', 'Mitotic_duration_bin',
       'Unique_Track_ID', 'Unique_Source_ID', 'Laggings',
       'Massive Missegregation', 'DNA bridge', 'Misaligned', 'MN',
       'Multipolar Divition', 'Citokinesis Failure', 'Slippage',
       'Mitotic Death', 'Comments', 'any_true', 'Daughter_dividing?',
       'Mother_divides_in_time'],
      dtype='object')

In [676]:
df_filtered[df_filtered["Mother_divides_in_time"] == True].any_true.value_counts()

any_true
False    33
Name: count, dtype: int64

In [677]:
df[df["Mother_divides_in_time"] == True].any_true.value_counts()

any_true
False    33
True      3
Name: count, dtype: int64

In [678]:
cols = [
    "Laggings", 
    "Massive Missegregation", 
    "DNA bridge", 
    "Misaligned", 
    "Multipolar Divition", 
    'Citokinesis Failure',
    'Slippage',
    'Mitotic Death'
       ]

df_short = df[cols + ['Dataset', 'Metaphase_arrested']]

# Work on a copy to avoid warnings
#bool_df = df_short.loc[:, cols].copy()

# Convert and cast to nullable boolean
df_short[cols] = df_short.loc[:, cols].apply(lambda col: col.astype("boolean"))

print(df_short.dtypes)

Laggings                  boolean
Massive Missegregation    boolean
DNA bridge                boolean
Misaligned                boolean
Multipolar Divition       boolean
Citokinesis Failure       boolean
Slippage                  boolean
Mitotic Death             boolean
Dataset                     int64
Metaphase_arrested         object
dtype: object


/var/folders/lr/cb1nd6l97696j8chvkss86580000gn/T/ipykernel_5082/3112547800.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_short[cols] = df_short.loc[:, cols].apply(lambda col: col.astype("boolean"))


In [679]:
bool_cols = df_short.select_dtypes(include = bool).columns
grouped = df_short.groupby(['Dataset', 'Metaphase_arrested'])[bool_cols].mean().mul(100)
grouped['Normal'] = 100 - grouped.sum(axis=1)

mean_df = grouped.groupby('Metaphase_arrested').mean()
sem_df  = grouped.groupby('Metaphase_arrested').sem().fillna(0)

plot_df = (
    mean_df.reset_index().melt(id_vars = 'Metaphase_arrested', var_name = 'Category', value_name = 'Mean')
    .merge(
        sem_df.reset_index().melt(id_vars = 'Metaphase_arrested', var_name = 'Category', value_name = 'SEM'),
        on = ['Metaphase_arrested', 'Category']
    )
)

# Stacking order
order = [c for c in mean_df.columns if c != 'Normal'] + ['Normal']
plot_df['Category'] = pd.Categorical(plot_df['Category'], categories = order, ordered = True)

# sort by condition then the category order so cumsum matches visual order
plot_df['order_index'] = plot_df['Category'].cat.codes
plot_df = plot_df.sort_values(['Metaphase_arrested', 'order_index']).reset_index(drop = True)

# compute bottoms & tops (explicit stacking)
plot_df['bottom'] = plot_df.groupby('Metaphase_arrested')['Mean'].cumsum() - plot_df['Mean']
plot_df['top']    = plot_df['bottom'] + plot_df['Mean']

# compute error bar extents relative to each slice
plot_df['err_low']  = plot_df['bottom'] + (plot_df['Mean'] - plot_df['SEM'])
plot_df['err_high'] = plot_df['bottom'] + (plot_df['Mean'] + plot_df['SEM'])

# Optionally clamp lower error to bottom (uncomment if you want that):
# plot_df['err_low'] = plot_df[['err_low', 'bottom']].max(axis=1)

# build charts using precomputed bottoms/tops (no implicit stacking)
bars = alt.Chart(plot_df).mark_bar(size = 25).encode(
    x = alt.X('Metaphase_arrested:N', title = 'Mother cell mitotic duration'),
    y = alt.Y('bottom:Q', title = 'Percentage (%)', scale = alt.Scale(domain = [0, 100])),
    y2 = 'top:Q',
    color = alt.Color('Category:N', sort = order, legend = alt.Legend(title = 'Category')),
    tooltip = ['Metaphase_arrested:N', 'Category:N', alt.Tooltip('Mean:Q', format = '.2f'), alt.Tooltip('SEM:Q', format = '.2f')]
).properties(width = 200, height = 200)

# error rules placed at the same horizontal position; colored by category (no legend duplication)
err = alt.Chart(plot_df).mark_rule(strokeWidth = 2).encode(
    x = 'Metaphase_arrested:N',
    y = 'err_low:Q',
    y2 = 'err_high:Q',
    color = alt.Color('Category:N', sort = order, legend = None),
    tooltip = ['Metaphase_arrested:N', 'Category:N', alt.Tooltip('Mean:Q', format = '.2f'), alt.Tooltip('SEM:Q', format = '.2f')]
)

chart = (bars + err).properties(
    #title='Abnormalities & Normal per Condition (mean ± SEM across datasets)'
)

chart

alt.LayerChart(...)